### Contexto e Objetivo

Está sendo desenvolvido um modelo de classificação de dados em uma base de serviço para avaliar se um cliente irá continuar no serviço ou irá cancelá-lo. Essa analise poderá ser util para identificar as chances do cliente querer desistir do serviço que possui, e avaliar os fatores que podem afetar nessa decisão


Importação de Dependências

In [27]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import numpy as np 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

In [28]:
df_credit = pd.read_csv(r"..\\documents\\Credit_Card_Churn.csv")

# Análise Inicial da base de Crédito

In [29]:
df_credit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CLIENTNUM                 10000 non-null  int64  
 1   Attrition_Flag            10000 non-null  object 
 2   Customer_Age              10000 non-null  int64  
 3   Gender                    10000 non-null  object 
 4   Dependent_count           10000 non-null  int64  
 5   Education_Level           10000 non-null  object 
 6   Marital_Status            10000 non-null  object 
 7   Income_Category           10000 non-null  object 
 8   Card_Category             10000 non-null  object 
 9   Months_on_book            10000 non-null  int64  
 10  Total_Relationship_Count  10000 non-null  int64  
 11  Months_Inactive_12_mon    10000 non-null  int64  
 12  Contacts_Count_12_mon     10000 non-null  int64  
 13  Credit_Limit              10000 non-null  float64
 14  Total_R

Análise de valores únicos

In [30]:
for x in df_credit.columns:
    print(f'Coluna: {x}')
    print(np.sort(df_credit[x].unique()), "\n")
    

Coluna: CLIENTNUM
[100000000 100000001 100000002 ... 100009997 100009998 100009999] 

Coluna: Attrition_Flag
['Attrited Customer' 'Existing Customer'] 

Coluna: Customer_Age
[26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49
 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72] 

Coluna: Gender
['F' 'M'] 

Coluna: Dependent_count
[0 1 2 3 4 5] 

Coluna: Education_Level
['College' 'Doctorate' 'Graduate' 'High School' 'Post-Graduate'
 'Uneducated'] 

Coluna: Marital_Status
['Divorced' 'Married' 'Single'] 

Coluna: Income_Category
['$120K +' '$40K - $60K' '$60K - $80K' '$80K - $120K' 'Less than $40K'
 'Unknown'] 

Coluna: Card_Category
['Blue' 'Gold' 'Platinum' 'Silver'] 

Coluna: Months_on_book
[12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35
 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59
 60] 

Coluna: Total_Relationship_Count
[1 2 3 4 5 6] 

Coluna: Months_Inactive_12_mon
[0 1 2 3 4 5 6] 

Coluna: 

Avaliar se nos valores numericos apresentam valores abaixo de 0

In [31]:
colunas_numericas = [x for x in df_credit.columns if df_credit[x].dtype != 'object']

for x in colunas_numericas:
    print(x, ";", (df_credit[x] < 0).any())


CLIENTNUM ; False
Customer_Age ; False
Dependent_count ; False
Months_on_book ; False
Total_Relationship_Count ; False
Months_Inactive_12_mon ; False
Contacts_Count_12_mon ; False
Credit_Limit ; False
Total_Revolving_Bal ; False
Avg_Open_To_Buy ; True
Total_Amt_Chng_Q4_Q1 ; False
Total_Trans_Amt ; False
Total_Trans_Ct ; False
Total_Ct_Chng_Q4_Q1 ; False
Avg_Utilization_Ratio ; False


In [32]:
pd.unique(
    df_credit['Avg_Open_To_Buy'][df_credit['Avg_Open_To_Buy'] < 0]
)

array([ -215.88,  -144.15,  -328.15,  -610.36,  -608.72,  -819.11,
       -1153.61,  -288.25,  -635.31,  -486.53,   -52.3 ,  -258.55,
        -590.21,  -383.17,  -156.2 ,  -436.45,  -505.25,  -259.98,
        -413.29,  -465.85,  -767.9 ,  -581.34,  -343.26,  -438.38,
        -448.14,   -20.38,  -280.65,  -522.06,  -134.21,  -647.44,
        -975.79,  -756.8 , -1305.96, -1296.12,  -546.03,  -117.27,
        -634.48,  -460.49,  -423.37,  -157.11,  -789.05,  -480.35,
        -394.85, -1245.82,   -42.87,  -216.04,  -758.83,  -165.5 ,
        -129.32,  -760.76,  -726.5 ,  -362.22,  -291.37, -1233.57,
        -532.79,  -461.66,  -239.54,  -416.1 ,  -962.01,  -231.85,
        -887.8 ,  -527.11,  -641.07,  -225.11,  -571.53,  -172.98,
        -176.51, -1154.36,  -157.88,   -62.46,  -126.13,  -509.61,
        -943.43,  -545.01,  -385.57,  -787.08,  -732.01,  -246.04,
        -304.91,   -86.6 , -1088.64,  -728.6 ,  -166.26,  -524.  ,
        -815.16,   -38.63, -1268.82,  -211.46,  -822.45,  -405

In [33]:
df_credit[df_credit['Avg_Open_To_Buy'] < 0]

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
73,100000073,Existing Customer,29,F,4,Uneducated,Single,Less than $40K,Blue,31,4,2,1,2236.46,2452.34,-215.88,2.11,18624,25,0.45,1.10
140,100000140,Attrited Customer,52,M,4,High School,Married,$80K - $120K,Blue,53,1,5,6,2338.37,2482.52,-144.15,0.37,3813,109,1.00,1.06
207,100000207,Existing Customer,51,M,3,High School,Married,Less than $40K,Blue,31,5,2,4,1819.75,2147.90,-328.15,2.36,1917,90,0.58,1.18
209,100000209,Existing Customer,33,M,3,High School,Single,$80K - $120K,Blue,13,6,6,5,1322.28,1932.64,-610.36,0.36,19912,60,2.35,1.46
255,100000255,Existing Customer,32,M,3,College,Single,$60K - $80K,Blue,32,6,6,6,1263.89,1872.61,-608.72,1.05,11219,52,2.11,1.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9636,100009636,Attrited Customer,62,F,5,Doctorate,Married,$80K - $120K,Blue,52,6,5,3,1108.46,1308.83,-200.37,2.37,5385,76,2.03,1.18
9674,100009674,Existing Customer,41,F,4,Doctorate,Married,Less than $40K,Blue,22,6,6,3,1062.44,1317.75,-255.31,1.27,4943,70,2.91,1.24
9796,100009796,Attrited Customer,45,M,4,Uneducated,Married,$40K - $60K,Blue,19,1,6,0,1517.64,2076.39,-558.75,1.85,16485,76,2.38,1.37
9950,100009950,Existing Customer,60,F,3,College,Married,$80K - $120K,Blue,51,1,4,6,1478.83,1571.78,-92.95,1.56,5193,73,0.92,1.06


In [25]:
# Análise percentual de clientes inexistentes

df_credit['Attrition_Flag'].value_counts(normalize=True)

Attrition_Flag
Existing Customer    0.85
Attrited Customer    0.15
Name: proportion, dtype: float64

In [ ]:
df_credit['Card_Category'].unique()

Remoção das variáveis irrelevantes

In [ ]:
# CLIENTNUM (pelo fato do CLIENTNUM ser um valor unico a sua inclusão no modelo de dados poderá afetar na prevenção de valores)

# Além disso foram retiradas as variaveis em string para não afetar o modelo

df_credit = df_credit.drop(columns=['CLIENTNUM' 'Gender', 'Education_Level', 
                                    'Marital_Status', 'Income_Category', 'Card_Category'])

df_credit

,Attrition_Flag,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,Existing Customer,57,0,51,1,2,6,3206.56,1878.94,1327.62,1.07,13740,34,2.09,0.59
1,Attrited Customer,61,3,19,2,2,6,5134.84,2498.54,2636.30,2.65,14279,64,0.61,0.49
2,Existing Customer,62,0,31,2,6,3,20704.64,1581.42,19123.22,2.00,19353,41,1.36,0.08
3,Existing Customer,39,3,20,5,0,5,28157.67,552.70,27604.97,2.56,18360,123,0.71,0.02
4,Existing Customer,52,3,24,1,5,0,27955.43,1430.73,26524.70,0.60,1205,69,0.46,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Attrited Customer,33,1,42,2,1,3,14118.49,1608.60,12509.89,1.33,12270,76,2.71,0.11
9996,Attrited Customer,69,3,39,6,0,5,1489.53,2139.22,-649.69,1.33,9976,20,1.19,1.44
9997,Attrited Customer,46,2,49,1,3,0,18717.68,398.80,18318.88,1.09,4585,43,0.57,0.02
9998,Existing Customer,33,3,16,5,3,1,24765.71,942.11,23823.60,2.44,17092,54,0.85,0.04


# Modelo de Regressão Logística

In [50]:
# Separação dos dados
X = df_credit.drop(columns=['Attrition_Flag'])
y = df_credit['Attrition_Flag']

In [51]:
# Divisão dos dados
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.3)

# Modelo Logistic Regression
logistic_regression = LogisticRegression()
logistic_regression.fit(X_train, Y_train)

prob_lr = logistic_regression.predict(X_test)
prob_lr

c:\Users\rpf97\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array(['Existing Customer', 'Existing Customer', 'Existing Customer', ...,
       'Existing Customer', 'Existing Customer', 'Existing Customer'],
      shape=(3000,), dtype=object)

In [52]:
logistic_regression.score(X, y)

0.8462